In [ ]:
# DADDI ADDOUN Sami
# Matrix number: 24223007
# CODE 1: CNN BASELINE

import numpy as np
import matplotlib.pyplot as plt
import os
import json
import cv2
import sys
import pickle
import time
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras.backend as K

# --- CONFIGURATION ---
BASE_PATH = 'YOUR_PATH_TO_DATA'
IMAGE_PATH = os.path.join(BASE_PATH, 'VQA_RAD_Image_Folder')
JSON_PATH = os.path.join(BASE_PATH, 'VQA_RAD_Dataset_Public.json')
RESULTS_PATH = 'YOUR_PATH_TO_RESULTS'

if not os.path.exists(RESULTS_PATH): os.makedirs(RESULTS_PATH)

IMG_SIZE = (224, 224)
NUM_ANS_TOP = 15
N_FOLDS = 5

# --- DATA LOADING ---
print("--- Loading CNN Data ---")
with open(JSON_PATH, 'r') as f: raw_data = json.load(f)

# Top 15 most frequent labels
counts = {}
for item in raw_data:
    ans = str(item['answer']).lower()
    counts[ans] = counts.get(ans, 0) + 1
top_ans = [x[0] for x in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:NUM_ANS_TOP]]
atoi = {w: i for i, w in enumerate(top_ans)}

# Load Data & Metadata
X_images, Y_ids = [], []
filenames, questions = [], []

print("Reading images...")
for item in tqdm(raw_data):
    ans = str(item['answer']).lower()
    if ans in atoi:
        path = os.path.join(IMAGE_PATH, item['image_name'])
        img = cv2.imread(path)
        if img is not None:
            img = cv2.resize(img, IMG_SIZE)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            X_images.append(img)
            Y_ids.append(atoi[ans])
            # Metadata collection
            filenames.append(item['image_name'])
            questions.append(str(item['question']).lower())

X_images = preprocess_input(np.array(X_images))
Y_ids = np.array(Y_ids)
Y_one_hot = to_categorical(Y_ids, num_classes=NUM_ANS_TOP)
filenames = np.array(filenames)
questions = np.array(questions)

# --- MODEL DEFINITION ---
def build_cnn_baseline():
    inp = Input(shape=(224, 224, 3))
    base = ResNet50(weights='imagenet', include_top=False, input_tensor=inp)
    # Freezing the base layers
    for layer in base.layers: layer.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(NUM_ANS_TOP, activation='softmax')(x)
    return Model(inp, out)

# --- CROSS-VALIDATION ---
kfold = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
all_true, all_pred = [], []
all_filenames, all_questions = [], []
inference_times = []
histories = []

print(f"\n--- Starting CNN CV ---")

fold = 1
for train_idx, val_idx in kfold.split(X_images, Y_ids):
    print(f"\nFold {fold}/{N_FOLDS}...")
    K.clear_session()

    # Data splitting
    X_train, X_val = X_images[train_idx], X_images[val_idx]
    Y_train, Y_val = Y_one_hot[train_idx], Y_one_hot[val_idx]

    # Metadata for this fold
    val_fnames = filenames[val_idx]
    val_qs = questions[val_idx]

    model = build_cnn_baseline()
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    es = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

    # Training the model

    h = model.fit(X_train, Y_train, validation_data=(X_val, Y_val),
              epochs=25, batch_size=32, callbacks=[es], verbose=1)


    histories.append(h.history)

    # Inference Time Measurement (with Warm-up)
    print("Measuring inference time...")
    # Warm-up step to initialize GPU (avoids cold start bias)
    _ = model.predict(X_val[:1], verbose=0)

    start_time = time.time()
    preds_probs = model.predict(X_val, verbose=0)
    end_time = time.time()

    # Calculate average time per sample in milliseconds
    avg_time_per_sample = ((end_time - start_time) / len(X_val)) * 1000
    inference_times.append(avg_time_per_sample)

    preds = np.argmax(preds_probs, axis=1)
    trues = np.argmax(Y_val, axis=1)

    all_pred.extend(preds)
    all_true.extend(trues)
    all_filenames.extend(val_fnames)
    all_questions.extend(val_qs)

    fold += 1

# --- SAVING RESULTS ---
# Calculate Model Size (Parameters)
model_params = model.count_params()

global_acc = accuracy_score(all_true, all_pred)
macro_f1 = f1_score(all_true, all_pred, average='macro')
mean_inference_time = np.mean(inference_times)

# Open vs Closed Questions Analysis
closed_ans = ['yes', 'no']
is_closed = [top_ans[i] in closed_ans for i in all_true]
is_closed = np.array(is_closed)
acc_closed = accuracy_score(np.array(all_true)[is_closed], np.array(all_pred)[is_closed])
acc_open = accuracy_score(np.array(all_true)[~is_closed], np.array(all_pred)[~is_closed])

results_data = {
    'model': 'CNN_Baseline',
    'accuracy': global_acc,
    'macro_f1': macro_f1,
    'acc_closed': acc_closed,
    'acc_open': acc_open,
    'all_true': all_true,
    'all_pred': all_pred,
    'classes': top_ans,
    'filenames': all_filenames,
    'questions': all_questions,
    'history': histories,
    'inference_time_ms': mean_inference_time,
    'params': model_params
}

save_file = os.path.join(RESULTS_PATH, 'cnn_results.pkl')
with open(save_file, 'wb') as f:
    pickle.dump(results_data, f)

print(f"\n CNN Results saved to: {save_file}")
print(f"\n Average Inference Time: {mean_inference_time:.2f} ms/sample")

--- Loading CNN Data ---
Reading images...


100%|██████████| 2248/2248 [01:27<00:00, 25.69it/s] 



--- Starting CNN CV ---

Fold 1/5...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/25
35/35 ━━━━━━━━━━━━━━━━━━━━ 31s 504ms/step - accuracy: 0.4028 - loss: 2.4710 - val_accuracy: 0.5109 - val_loss: 1.2769
Epoch 2/25
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 107ms/step - accuracy: 0.5409 - loss: 1.3056 - val_accuracy: 0.5652 - val_loss: 1.1743
Epoch 3/25
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 103ms/step - accuracy: 0.5811 - loss: 1.1231 - val_accuracy: 0.5543 - val_loss: 1.1754
Epoch 4/25
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 106ms/step - accuracy: 0.5992 - loss: 1.0945 - val_accuracy: 0.5399 - val_loss: 1.2039
Epoch 5/25
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 104ms/step - accuracy: 0.6197 - loss: 1.0414 - val_accuracy: 0.5725 - val_loss: 1.1762
Epoch 6/25
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - accuracy: 0.6219 - loss: 1.0134 - val_accuracy: 0.5725 - val_loss: 1.1746
Measuring inference time...

Fold 2/5...
Epoch 1/25
35/35 ━━━━━━━━━━━━━━━━━━━━ 29s 560ms/step - accuracy: 0.4061 - loss: 2.5373 - val_accuracy